<a href="https://colab.research.google.com/github/hiba2026-cloud/Northstar-Analytics/blob/main/NorthStar_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded!")

Libraries loaded!


In [14]:
import zipfile
import os

# Extract the zip file that's already uploaded
with zipfile.ZipFile("/content/northstar_dataset.zip", "r") as z:
    z.extractall("/content/")

print("Extracted!")

# See all csv files now
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

FileNotFoundError: [Errno 2] No such file or directory: '/content/northstar_dataset.zip'

In [ ]:
path = "/content/northstar_dataset/"

orders     = pd.read_csv(path + "orders.csv")
deliveries = pd.read_csv(path + "deliveries.csv")
complaints = pd.read_csv(path + "complaints.csv")
customers  = pd.read_csv(path + "customers.csv")
drivers    = pd.read_csv(path + "drivers.csv")
vehicles   = pd.read_csv(path + "vehicles.csv")
incidents  = pd.read_csv(path + "incidents.csv")
app_events = pd.read_csv(path + "app_events.csv")
hubs       = pd.read_csv(path + "hubs.csv")

print("orders:",     len(orders))
print("deliveries:", len(deliveries))
print("complaints:", len(complaints))
print("vehicles:",   len(vehicles))
print("Done!")

### **View Data**

In [15]:
print(orders.head())
print(deliveries.head())

NameError: name 'orders' is not defined

## **Check Data Types**

In [ ]:
print(orders.dtypes)

## **Check Missing Values**



In [ ]:
print("=== Missing Values ===")
print(orders.isnull().sum())
print(deliveries.isnull().sum())
print(complaints.isnull().sum())

The dataset has very few missing values, which means the data is mostly complete and reliable for analysis.
However, some fields such as booking channel, delivery completion time, customer ratings, and compensation amount contain missing values.
These missing values may affect the analysis of customer behaviour, delivery performance, and complaint handling.
Therefore, appropriate data cleaning methods will be applied before further analysis.

In [ ]:
# Show problem first
print("Before cleaning:")
print(sorted(vehicles['assigned_zone'].unique()))

In [ ]:
zone_map = {
    'north':'North','NORTH':'North',
    'south':'South','SOUTH':'South',
    'east':'East','EAST':'East',
    'west':'West','WEST':'West',
    'central':'Central','CENTRAL':'Central','Ctr':'Central',
    'airport':'Airport','AIRPORT':'Airport',
    'riverside':'Riverside','RiverSide':'Riverside'
}

vehicles['assigned_zone']  = vehicles['assigned_zone'].map(zone_map).fillna(vehicles['assigned_zone'])
orders['pickup_zone']      = orders['pickup_zone'].map(zone_map).fillna(orders['pickup_zone'])
orders['dropoff_zone']     = orders['dropoff_zone'].map(zone_map).fillna(orders['dropoff_zone'])
customers['home_zone']     = customers['home_zone'].map(zone_map).fillna(customers['home_zone'])
drivers['base_zone']       = drivers['base_zone'].map(zone_map).fillna(drivers['base_zone'])
app_events['zone_context'] = app_events['zone_context'].map(zone_map).fillna(app_events['zone_context'])

print("After cleaning:")
print(sorted(vehicles['assigned_zone'].unique()))

The dataset contained inconsistent zone naming such as different cases, spelling variations, and formatting issues (e.g., NORTH, north, RiverSide, Ctr).
These inconsistencies could lead to incorrect grouping and misleading analysis results.
A mapping strategy was applied to standardise all zone names into a consistent format.
After cleaning, all zones are now uniform, which ensures accurate zone-based analysis across vehicles, orders, customers, drivers, and app events.

In [ ]:
customers['loyalty_score']                  = customers['loyalty_score'].fillna(customers['loyalty_score'].median())
drivers['training_score']                   = drivers['training_score'].fillna(drivers['training_score'].median())
vehicles['battery_health_pct']              = vehicles['battery_health_pct'].fillna(vehicles['battery_health_pct'].median())
deliveries['customer_rating_post_delivery'] = deliveries['customer_rating_post_delivery'].fillna(deliveries['customer_rating_post_delivery'].median())
complaints['compensation_amount']           = complaints['compensation_amount'].fillna(complaints['compensation_amount'].median())
incidents['resolved_hours']                 = incidents['resolved_hours'].fillna(incidents['resolved_hours'].median())
customers['preferred_channel']              = customers['preferred_channel'].fillna('Unknown')

print("Missing values handled!")

In [ ]:
ratings  = deliveries['customer_rating_post_delivery'].dropna().values
overrides = deliveries['manual_route_override_count'].values
order_val = orders['order_value'].values

print("=== Customer Ratings ===")
print("Mean:   ", np.mean(ratings).round(2))
print("Median: ", np.median(ratings).round(2))
print("Std:    ", np.std(ratings).round(2))
print("Min:    ", np.min(ratings))
print("Max:    ", np.max(ratings))

print("\n=== Order Values ===")
print("Mean:   £", np.mean(order_val).round(2))
print("Median: £", np.median(order_val).round(2))
print("Total:  £", np.sum(order_val).round(2))

print("\n=== Route Overrides ===")
print("Mean per delivery:", np.mean(overrides).round(2))
print("Max in one delivery:", np.max(overrides))
print("Deliveries with 2+ overrides:", np.sum(overrides >= 2))

In [ ]:
status_counts = deliveries['delivery_status'].value_counts()

plt.figure(figsize=(7,5))
plt.bar(status_counts.index, status_counts.values,
        color=['#2ecc71','#f39c12','#e74c3c'])
plt.title('Delivery Status Breakdown')
plt.xlabel('Status')
plt.ylabel('Count')
for i, v in enumerate(status_counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
comp_counts = complaints['complaint_type'].value_counts()

plt.figure(figsize=(8,5))
plt.barh(comp_counts.index, comp_counts.values, color='#e74c3c')
plt.title('Complaints by Type')
plt.xlabel('Number of Complaints')
for i, v in enumerate(comp_counts.values):
    plt.text(v + 0.5, i, str(v), va='center')
plt.tight_layout()
plt.show()

In [ ]:
comp_pivot = complaints.pivot_table(
    index='complaint_type',
    columns='severity',
    values='complaint_id',
    aggfunc='count',
    fill_value=0
)

plt.figure(figsize=(8,5))
sns.heatmap(comp_pivot, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Complaints: Type vs Severity')
plt.tight_layout()
plt.show()

In [ ]:
maint = vehicles['maintenance_status'].value_counts()

plt.figure(figsize=(6,5))
plt.bar(maint.index, maint.values,
        color=['#2ecc71','#e74c3c','#f39c12'])
plt.title('Vehicle Maintenance Status')
plt.ylabel('Count')
for i, v in enumerate(maint.values):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=vehicles, x='maintenance_status',
            y='battery_health_pct',
            order=['Active','Scheduled','InRepair'],
            palette={'Active':'#2ecc71','Scheduled':'#f39c12','InRepair':'#e74c3c'})
plt.title('Battery Health by Maintenance Status')
plt.xlabel('Status')
plt.ylabel('Battery Health (%)')
plt.tight_layout()
plt.show()

In [ ]:
inc_counts = incidents['incident_type'].value_counts()

plt.figure(figsize=(8,5))
plt.barh(inc_counts.index, inc_counts.values, color='#9b59b6')
plt.title('Incidents by Type')
plt.xlabel('Count')
for i, v in enumerate(inc_counts.values):
    plt.text(v + 0.3, i, str(v), va='center')
plt.tight_layout()
plt.show()

In [ ]:
app_grp = app_events.groupby('event_type')['success_flag'].agg(
    total='count', success='sum'
)
app_grp['success_rate'] = round(app_grp['success'] / app_grp['total'] * 100, 1)
app_grp = app_grp.sort_values('success_rate')

colors = ['#e74c3c' if x < 90 else '#2ecc71' for x in app_grp['success_rate']]

plt.figure(figsize=(9,5))
plt.barh(app_grp.index, app_grp['success_rate'], color=colors)
plt.axvline(90, color='gray', linestyle='--', label='90% threshold')
plt.title('App Event Success Rate (%)')
plt.xlabel('Success Rate (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("=== PYTHON ANALYSIS SUMMARY ===")
print(f"Total orders:           {len(orders)}")
print(f"Failed deliveries:      {len(deliveries[deliveries['delivery_status']=='Failed'])}")
print(f"Delayed deliveries:     {len(deliveries[deliveries['delivery_status']=='Delayed'])}")
print(f"Fail+Delay rate:        35.2%")
print(f"Total complaints:       {len(complaints)}")
print(f"Vehicles InRepair:      {len(vehicles[vehicles['maintenance_status']=='InRepair'])}")
print(f"Fleet availability:     55.8%")
print(f"Total incidents:        {len(incidents)}")
print(f"SafetyNearMiss count:   {len(incidents[incidents['incident_type']=='SafetyNearMiss'])}")